In [1]:
raw_text= """

=======================mon
53	13	48	9	83 [52-6p-25] [46]
53	4	9	79	22 [7-39p-55p] [46]
36	58	9	61	46 [7-39-48/31] [17]
66	11	34	22	77 [7-71-48p] [67]
55	8	52	67	48 [7-39-48] [70]
69	20	27	67	12 [78-39p-47] [67p]
76	12	42	14	8 [7-70-31] [70]
4	21	31	16	38 [57-6-25] [70]
57	41	53	20	64 [44-70/5-25] [46]
71	35	17	14	75 [7-71/6-25] [46p]
46	74	25	84	90 [52-6-25] [53]
25	36	32	83	77 [57-71p-55] [70p]
36	50	88	37	64 [57-48-55] [53] 
77	62	40	85	87 [78-39p-48] [46]
64	62	47	80	53 [78-39-55] [67/17]
64	62	47	80	53 [78-39-55] [67/17]
15	12	9	4	49 [7-71/70-31] [53]
32	21	56	14	57 [7-6-55] [67/46] 

============================tues

62	10	17	27	1 [52-70/71p-47] [53]
79	37	28	43	74 [57/44-6-25/55] [17p]
44	88	65	42	51 [78/7/44-6-31] [46p]
14	81	68	33	30 [44-6-47] [70p]
22	17	26	89	2 [78-6-48] [46p]
67	6	45	23	38 [57-39-31/25] [46p]
61	90	68	37	54 [52-39-31] [70]
46	79	71	53	62 [52-70-47] [53p]
45	57	59	8	64 [52p/78-71-47] [17]
17	63	52	46	78 [52/57-70-31] [67/46]
37	52	38	71	57 [57-71-31] [67/46]
13	20	72	76	71 [7-71-47/55] [17p]
50	74	6	48	67 [7-6-31] [67]
40	20	53	13	4 [78p-71-31] [67]
60	77	87	65	36 [7-70p-55] [46p]
38	34	86	25	23 [57-6-31] [46]
42	62	75	78	5 [52-6-55] [46]
86	10	72	61	21 [7-70-25] [53]

============================== wed
76	45	57	31	87 [52-71-55] [17]
49	31	33	54	23 [52-6p-31] [46]
88	57	47	41	87 [52/78-39-31] [67]
12	18	35	31	60 [57-39-48] [70]
75	44	21	14	40 [44-39-55/48] [46]
89	45	69	7	30 [78-70-48] [70/46]
3	24	13	68	44 [7-6p-31] [46]
18	60	26	63	39 [78-71-31] [17p]
60	43	82	48	11 [78-39-31] [17p]
13	90	67	87	16 [57-70-25p] [17p]
39	3	44	77	64 [7-71p-55] [53]
86	68	16	19	59 [7-6-48p] [70]
6	77	80	75	13 [7-6p-31p] [53]
25	23	8	88	22 [7-6p-31p] [53]
17	18	62	78	44 [57-6p-25] [67]
81	45	27	54	24 [57-39-55] [17]
25	63	47	49	75 [52-39-48/55] [53]
25	82	31	85	14 [78-70-31p/55] [67]
88	78	13	83	72 [44-6-31] [67]
42	26	65	40	6 [44-6-25] [67]
75	86	32	61	76 [78p-70-31] [46]
5	25	71	19	56 [78p-39-55] [67/17]
49	86	9	53	27 [44-70-25] [46p]
69	55	82	41	56 [57p/44-39-47] [70]
29	75	84	13	60 [78-6-31] [67p]
25	82	71	57	63 [57/78-6-47] [70]
30	39	70	6	29 [78-48-25] [46]
70	78	58	35	73 [57-39/70/6-47] [70]
40	87	49	15	42 [7-71-31] [67]
88	17	55	68	50 [44-71-47] [17]
60	89	12	81	41 [57p/7p-70-31] [17p]  
 
"""

In [2]:
import re
import json
from collections import defaultdict


# ---------------- PARSER ----------------
def parse_to_machine_json(text):
    machines = defaultdict(lambda: {"a": [], "b": [], "c": []})
    batch_index = defaultdict(int)

    for line in text.splitlines():
        line = line.strip()

        if not line or line.startswith("="):
            continue

        if not re.match(r'^\d', line):
            continue

        # first 5 numbers before bracket
        left = line.split('[')[0]
        nums = list(map(int, re.findall(r'\d+', left)))[:5]

        # last bracket = machine
        brackets = re.findall(r'\[(.*?)\]', line)
        if not brackets:
            continue

        machine_raw = brackets[-1]
        machine_ids = re.findall(r'\d+', machine_raw)

        for mid in machine_ids:
            idx = batch_index[mid] % 3
            key = ["a", "b", "c"][idx]
            machines[mid][key].extend(nums)
            batch_index[mid] += 1

    return machines


# ---------------- PRETTY JSON ----------------
def pretty_print_json(data):
    print("\n===== JSON OUTPUT =====")
    print(json.dumps(data, indent=2, sort_keys=True))


# ---------------- HUMAN VIEW ----------------
def pretty_print_machines(machines):
    print("\n===== HUMAN VIEW =====")
    for machine in sorted(machines.keys(), key=int):
        print(f"\nMachine {machine}")
        for section in ["a", "b", "c"]:
            nums = machines[machine][section]
            if nums:
                line = ", ".join(map(str, nums))
                print(f"  {section}: {line}")


# ---------------- INTERSECTIONS ----------------
def compute_intersections(machines):
    results = {}
    for machine in sorted(machines.keys(), key=int):
        a = set(machines[machine]['a'])
        b = set(machines[machine]['b'])
        c = set(machines[machine]['c'])
        results[machine] = {
            'a_n_b': sorted(a & b),
            'a_n_c': sorted(a & c),
            'b_n_c': sorted(b & c),
            'a_n_b_n_c': sorted(a & b & c),
        }
    return results

def pretty_print_intersections(intersections):
    print("\n===== INTERSECTIONS =====")
    for machine in sorted(intersections.keys(), key=int):
        vals = intersections[machine]
        print(f"\nMachine {machine} intersections:")
        print(f"  a ∩ b: {vals['a_n_b']}")
        print(f"  a ∩ c: {vals['a_n_c']}")
        print(f"  b ∩ c: {vals['b_n_c']}")
        print(f"  a ∩ b ∩ c: {vals['a_n_b_n_c']}")

# ---------------- RUN ----------------
if __name__ == "__main__":
    result = parse_to_machine_json(raw_text)

    pretty_print_json(result)
    pretty_print_machines(result)

    intersections = compute_intersections(result)
    pretty_print_intersections(intersections)



===== JSON OUTPUT =====
{
  "17": {
    "a": [
      36,
      58,
      9,
      61,
      46,
      79,
      37,
      28,
      43,
      74,
      76,
      45,
      57,
      31,
      87,
      13,
      90,
      67,
      87,
      16,
      88,
      17,
      55,
      68,
      50
    ],
    "b": [
      64,
      62,
      47,
      80,
      53,
      45,
      57,
      59,
      8,
      64,
      18,
      60,
      26,
      63,
      39,
      81,
      45,
      27,
      54,
      24,
      60,
      89,
      12,
      81,
      41
    ],
    "c": [
      64,
      62,
      47,
      80,
      53,
      13,
      20,
      72,
      76,
      71,
      60,
      43,
      82,
      48,
      11,
      5,
      25,
      71,
      19,
      56
    ]
  },
  "46": {
    "a": [
      53,
      13,
      48,
      9,
      83,
      71,
      35,
      17,
      14,
      75,
      44,
      88,
      65,
      42,
      51,
      17,
      63,
      52,
      46,
 